# Módulo de Análisis Multivariante (`heavystats.multivariante`)
## Pipeline en Dos Etapas para Biomonitoreo Toxicológico y Epidemiología

Este cuaderno demuestra el flujo bioestadístico y computacional de la librería **HeavyStats** para dar respuesta al **Objetivo Específico 4** de la investigación:

> **Objetivo 4**: Evaluar el impacto individual y conjunto de los factores de riesgo sobre las concentraciones biológicas de metales pesados ($\ce{Hg_T}$ en plasma/sangre).

### Desafíos Metodológicos Resueltos:
1. **Desbalance Dimensional ($p > n$):** Con $n = 20$ muestras analíticas y $p > 35$ factores de exposición, las regresiones clásicas colapsan por pérdida de grados de libertad e hiper-sobreajuste (*overfitting*).
2. **Asimetría No Paramétrica:** La variable biológica presenta asimetría positiva con valores concentrados cerca del límite de detección ($LOD$).
3. **Solución en Dos Etapas:**
   - **Etapa 1: Tamizaje Bivariante No Paramétrico (*Screening*)** con contrastes libres de supuestos (Mann-Whitney $U$, Kruskal-Wallis $H$, Spearman $\rho_s$) y control de la Tasa de Falso Descubrimiento (**Benjamini-Hochberg FDR**).
   - **Etapa 2: Modelado Multivariante Regularizado**, combinando **Regresión LASSO** (selección parsimoniosa de factores netos) y **Regresión PLS-R** (proyección en componentes latentes y cálculo de **VIP $\ge 1{,}0$**).

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

import heavystats as hs
from heavystats.multivariante import (
    MultivariatePipeline,
    MultivariateScreening,
    LassoModeler,
    PlsModeler,
    MultivariateTableReport
)

print(f"Versión de HeavyStats: {hs.__version__}")
print(f"Módulos multivariantes activos: {[cls for cls in dir(hs.multivariante) if not cls.startswith('_')]}")

Versión de HeavyStats: 0.4.4
Módulos multivariantes activos: ['LassoModeler', 'MultivariatePipeline', 'MultivariateScreening', 'MultivariateTableReport', 'PlsModeler', 'models', 'pipeline', 'prepare_features_matrix', 'reports', 'screening']


---
## 1. Carga y Estructuración de los Datos Analíticos de Mercurio ($n = 20$)
Cargamos la submuestra cuantitativa con determinación analítica de $\ce{Hg_T}$ en sangre total.

In [2]:
# Localizar el archivo de datos del proyecto
data_candidates = [
    Path(r"d:\vscode\results\resultados_mercurio\datos\procesados\datos_mercurio_n20.csv"),
    Path.cwd().parent / "datos" / "procesados" / "datos_mercurio_n20.csv",
    Path.cwd() / "datos" / "datos_mercurio_n20.csv",
]

data_path = None
for p in data_candidates:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("No se encontró el archivo de datos analíticos.")

df_hg = pd.read_csv(data_path)
print(f"Dataset analítico cargado exitosamente: {df_hg.shape[0]} observaciones y {df_hg.shape[1]} variables.")
print(f"Resumen de la variable respuesta (Mercurio_ug_L):")
print(df_hg['Mercurio_ug_L'].describe())

Dataset analítico cargado exitosamente: 20 observaciones y 56 variables.
Resumen de la variable respuesta (Mercurio_ug_L):
count    20.000000
mean      2.354664
std       0.846047
min       1.190668
25%       1.567671
50%       2.211717
75%       3.114953
max       3.798271
Name: Mercurio_ug_L, dtype: float64


Observamos las primeras filas de covariables candidatas (dietarias, ambientales y sociodemográficas):

In [ ]:
cols_muestra = ['Muestra_Codificada', 'Edad', 'Sexo', 'Mercurio_ug_L', 'Alim_Pescados', 'Salud_Transporte_vehiculo', 'Alim_Cereales',]
df_hg[cols_muestra].head(10)

,Muestra_Codificada,Edad,Sexo,Mercurio_ug_L,Alim_Pescados,Salud_Transporte_vehiculo,Alim_Cereales,Es_Expuesto
0,1,7,femenino,2.698679,0,1,4,NO
1,2,8,masculino,2.227426,1,0,4,NO
2,3,9,masculino,1.881840,3,1,3,SI
3,4,7,masculino,1.253501,0,1,4,SI
4,5,6,femenino,1.881840,2,1,3,SI
5,6,7,masculino,1.536254,0,0,3,NO
6,7,9,femenino,3.232767,1,1,2,SI
7,8,7,masculino,2.824347,2,1,3,SI
8,9,6,masculino,3.515519,3,1,4,NO
9,10,5,femenino,1.190668,0,0,4,NO


---
## 2. Etapa 1: Tamizaje Bivariante No Paramétrico (*Screening*)

La clase `MultivariateScreening` evalúa sistemáticamente cada covariable independiente:
- **Variables dicotómicas (2 grupos):** Prueba $U$ de Mann-Whitney, estimador de Hodges-Lehmann y correlación biserial por rangos ($r_{\text{rb}}$ con intervalo de confianza *bootstrap* al 95%).
- **Variables politómicas ($\ge 3$ grupos):** Prueba de Kruskal-Wallis ($H$) y tamaño de efecto Épsilon Cuadrado ($\epsilon^2$).
- **Variables continuas / ordinales:** Coeficiente de correlación de Spearman ($\rho_s$) y Kendall ($\tau_b$).
- **Control de Multiplicidad:** Corrección Benjamini-Hochberg FDR para derivar los $q$-valores.
- **Clasificación Multicriterio:** Clasificación en prioridades **Alta**, **Intermedia** y **Baja / Descarte**.

In [4]:
# Inicializar y ejecutar el tamizaje
screening = MultivariateScreening(
    data=df_hg,
    target_col="Mercurio_ug_L",
    alpha=0.05,
    fdr_threshold=0.10,
    n_boot=1000,
    random_state=42
)

df_screening = screening.run()

# Visualizar el top 12 de factores ordenados por significancia
cols_view = ['Variable', 'Etiqueta', 'Tipo', 'Estadístico_Str', 'p_valor', 'q_fdr', 'Direccion', 'Prioridad']
df_screening[cols_view].head(12)

,Variable,Etiqueta,Tipo,Estadístico_Str,p_valor,q_fdr,Direccion,Prioridad
0,Alim_Pescados,Pescados,Ordinal / Continua,ρ = +0.523,0.017935,0.717400,Aporte (↑),Alta
1,Salud_Transporte_vehiculo,Vehículo,Binaria,U = 54.0,0.041812,0.836236,Aporte (↑),Alta
2,Salud_Transporte_publico,Transporte Público,Binaria,U = 69.0,0.067847,0.904626,Aporte (↑),Intermedia
3,Alim_Cereales,Cereales,Ordinal / Continua,ρ = -0.371,0.107143,0.937444,Atenuación (↓),Intermedia
4,Salud_Fuma,Hábito de Fumar,Binaria,U = 28.0,0.177122,0.937444,Atenuación (↓),Intermedia
5,Edad,Edad,Ordinal / Continua,ρ = +0.311,0.181462,0.937444,Aporte (↑),Intermedia
6,Exposicion_Talleres_mecanico,Taller Mecánico,Binaria,U = 52.5,0.204780,0.937444,Aporte (↑),Intermedia
7,Exposicion_Cualquier_Taller,Cualquier Taller en Entorno,Binaria,U = 52.5,0.204780,0.937444,Aporte (↑),Intermedia
8,Alim_Lacteos,Lácteos,Ordinal / Continua,ρ = +0.275,0.241106,0.937444,Aporte (↑),Baja / Descarte
9,Alim_Azucar,Azúcar,Ordinal / Continua,ρ = -0.270,0.248774,0.937444,Atenuación (↓),Baja / Descarte


Analizamos la distribución de covariables según su nivel de prioridad para el modelado multivariante:

In [5]:
print("Distribución de covariables por nivel de prioridad:")
print(df_screening['Prioridad'].value_counts())

factores_priorizados = screening.get_prioritized_features(min_priority="Intermedia")
print(f"\nFactores seleccionados para la Etapa 2 ({len(factores_priorizados)} covariables):")
for f in factores_priorizados:
    row = df_screening[df_screening['Variable'] == f].iloc[0]
    print(f"  • {row['Etiqueta']:<35} ({row['Variable']}): {row['Estadístico_Str']}, p={row['p_valor']:.3f}, {row['Direccion']}")

Distribución de covariables por nivel de prioridad:
Prioridad
Baja / Descarte    29
Intermedia          9
Alta                2
Name: count, dtype: int64

Factores seleccionados para la Etapa 2 (11 covariables):
  • Pescados                            (Alim_Pescados): ρ = +0.523, p=0.018, Aporte (↑)
  • Vehículo                            (Salud_Transporte_vehiculo): U = 54.0, p=0.042, Aporte (↑)
  • Transporte Público                  (Salud_Transporte_publico): U = 69.0, p=0.068, Aporte (↑)
  • Cereales                            (Alim_Cereales): ρ = -0.371, p=0.107, Atenuación (↓)
  • Hábito de Fumar                     (Salud_Fuma): U = 28.0, p=0.177, Atenuación (↓)
  • Edad                                (Edad): ρ = +0.311, p=0.181, Aporte (↑)
  • Taller Mecánico                     (Exposicion_Talleres_mecanico): U = 52.5, p=0.205, Aporte (↑)
  • Cualquier Taller en Entorno         (Exposicion_Cualquier_Taller): U = 52.5, p=0.205, Aporte (↑)
  • Uso de Joyería                    

---
## 3. Etapa 2: Modelado Multivariante Regularizado (LASSO y PLS-R)

### 3.1. Regresión LASSO ($L_1$ Penalizado)
Minimiza el error cuadrático incorporando una penalización sobre la norma $L_1$ de los coeficientes:
$$\min_{\beta} \left\{ \frac{1}{2n} \sum_{i=1}^n \left( y_i - \beta_0 - \sum_{j=1}^p \beta_j x_{ij} \right)^2 + \lambda \sum_{j=1}^p |\beta_j| \right\}$$

El hiperparámetro de penalización $\lambda$ se calibra mediante validación cruzada ($k$-fold CV con $k=5$).

In [6]:
from heavystats.multivariante.models import prepare_features_matrix

# Preparar matriz de predictores numéricos estandarizados
X_mat, feat_names = prepare_features_matrix(df_hg, factores_priorizados)
y_log = np.log(df_hg['Mercurio_ug_L'].values)

# Ajustar LassoModeler
lasso = LassoModeler(cv_folds=5, random_state=42)
lasso.fit(X_mat, y_log)

print(f"Parámetro de regularización óptimo (lambda/alpha): {lasso.alpha_:.4f}")
print(f"Coeficiente de determinación R² del modelo LASSO: {lasso.r2_:.3f}")
print(f"\nResumen de coeficientes estandarizados beta:")
print(lasso.summary_df.to_string(index=False))

Parámetro de regularización óptimo (lambda/alpha): 0.0870
Coeficiente de determinación R² del modelo LASSO: 0.290

Resumen de coeficientes estandarizados beta:
                    Variable  Beta_Estandarizado       Efecto_Lasso  Retenido
               Alim_Pescados            0.060000         Aporte (↑)      True
   Salud_Transporte_vehiculo            0.035466         Aporte (↑)      True
    Salud_Transporte_publico            0.017040         Aporte (↑)      True
                        Edad            0.000000 Descartado (β = 0)     False
Exposicion_Talleres_mecanico            0.000000 Descartado (β = 0)     False
 Exposicion_Cualquier_Taller            0.000000 Descartado (β = 0)     False
               Salud_Joyeria           -0.000000 Descartado (β = 0)     False
    Salud_Agua_pozo_profundo           -0.000000 Descartado (β = 0)     False
 Salud_Cualquier_Agua_Riesgo           -0.000000 Descartado (β = 0)     False
                  Salud_Fuma           -0.005242     Atenuac

### 3.2. Regresión por Mínimos Cuadrados Parciales (PLS-R) y Estadístico VIP
Modela la covarianza entre la matriz $X$ y el vector $y$ proyectándolos hacia $A = 2$ componentes latentes ortogonales.
La importancia de cada predictor se cuantifica mediante el estadístico **VIP** (Variable Importance in Projection):
$$\text{VIP}_j = \sqrt{ \frac{p \sum_{a=1}^A \text{SS}_a (w_{aj} / \|w_a\|)^2}{\sum_{a=1}^A \text{SS}_a} }$$
**Regla toxicológica:** Variables con $\text{VIP} \ge 1{,}0$ son determinantes críticos de la carga xenobiótica.

In [7]:
# Ajustar PlsModeler con 2 componentes latentes
pls = PlsModeler(n_components=2, scale=True)
pls.fit(X_mat, y_log)

print(f"Coeficiente de determinación R² del modelo PLS-R: {pls.r2_:.3f}")
print(f"\nResumen de importancia de variables en la proyección (VIP):")
print(pls.summary_df.to_string(index=False))

Coeficiente de determinación R² del modelo PLS-R: 0.432

Resumen de importancia de variables en la proyección (VIP):
                    Variable      VIP  Coef_PLS            Influencia_PLS  Es_Critico
               Alim_Pescados 1.313233  0.067538       Crítica (VIP ≥ 1.0)        True
   Salud_Transporte_vehiculo 1.289888  0.072719       Crítica (VIP ≥ 1.0)        True
               Alim_Cereales 1.143056 -0.060586       Crítica (VIP ≥ 1.0)        True
                  Salud_Fuma 1.124345 -0.077956       Crítica (VIP ≥ 1.0)        True
    Salud_Transporte_publico 1.108500  0.060029       Crítica (VIP ≥ 1.0)        True
Exposicion_Talleres_mecanico 1.015011  0.002909       Crítica (VIP ≥ 1.0)        True
 Exposicion_Cualquier_Taller 1.015011  0.002909       Crítica (VIP ≥ 1.0)        True
                        Edad 0.969050  0.022886 Moderada/Baja (VIP < 1.0)       False
               Salud_Joyeria 0.629406 -0.021672 Moderada/Baja (VIP < 1.0)       False
 Salud_Cualquier_Agua_R

---
## 4. Ejecución del Pipeline Integral Unificado (`MultivariatePipeline`)

En la práctica, todas las etapas anteriores se orquestan en una única llamada a la clase `MultivariatePipeline`:

In [8]:
pipeline = MultivariatePipeline(
    data=df_hg,
    target_col="Mercurio_ug_L",
    log_transform_target=True,
    alpha=0.05,
    fdr_threshold=0.10,
    random_state=42
)

# 1. Ejecutar tamizaje
pipeline.run_screening()

# 2. Ajustar modelos multivariantes sobre factores priorizados
fit_res = pipeline.fit_models(
    min_priority="Intermedia",
    lasso_cv_folds=5,
    pls_components=2
)

# 3. Obtener reporte consolidado
report = pipeline.get_report()
print("Pipeline ejecutado exitosamente. Tabla consolidada generada.")
report.df

Pipeline ejecutado exitosamente. Tabla consolidada generada.


,Variable,Etiqueta,Grupo,Tipo,Estadístico,Estadístico_Str,p_valor,q_fdr,Direccion_Biv,Prioridad,Beta_Estandarizado,Efecto_Lasso,VIP,Coef_PLS,Influencia_PLS
0,Alim_Pescados,Pescados,Frecuencia de Consumo Dietario (Ordinal 0-4),Ordinal / Continua,0.523148,ρ = +0.523,0.017935,0.717400,Aporte (↑),Alta,0.060000,Aporte (↑),1.313233,0.067538,Crítica (VIP ≥ 1.0)
1,Salud_Transporte_vehiculo,Vehículo,Hábitos de Salud y Fuentes de Agua,Binaria,54.000000,U = 54.0,0.041812,0.836236,Aporte (↑),Alta,0.035466,Aporte (↑),1.289888,0.072719,Crítica (VIP ≥ 1.0)
2,Alim_Cereales,Cereales,Frecuencia de Consumo Dietario (Ordinal 0-4),Ordinal / Continua,-0.371162,ρ = -0.371,0.107143,0.937444,Atenuación (↓),Intermedia,-0.036149,Atenuación (↓),1.143056,-0.060586,Crítica (VIP ≥ 1.0)
3,Salud_Fuma,Hábito de Fumar,Hábitos de Salud y Fuentes de Agua,Binaria,28.000000,U = 28.0,0.177122,0.937444,Atenuación (↓),Intermedia,-0.005242,Atenuación (↓),1.124345,-0.077956,Crítica (VIP ≥ 1.0)
4,Salud_Transporte_publico,Transporte Público,Hábitos de Salud y Fuentes de Agua,Binaria,69.000000,U = 69.0,0.067847,0.904626,Aporte (↑),Intermedia,0.017040,Aporte (↑),1.108500,0.060029,Crítica (VIP ≥ 1.0)
5,Exposicion_Talleres_mecanico,Taller Mecánico,Factores de Exposición Ambiental y Talleres,Binaria,52.500000,U = 52.5,0.204780,0.937444,Aporte (↑),Intermedia,0.000000,Descartado (β = 0),1.015011,0.002909,Crítica (VIP ≥ 1.0)
6,Exposicion_Cualquier_Taller,Cualquier Taller en Entorno,Factores de Exposición Ambiental y Talleres,Binaria,52.500000,U = 52.5,0.204780,0.937444,Aporte (↑),Intermedia,0.000000,Descartado (β = 0),1.015011,0.002909,Crítica (VIP ≥ 1.0)
7,Edad,Edad,Variables Sociodemográficas y Antropométricas,Ordinal / Continua,0.311355,ρ = +0.311,0.181462,0.937444,Aporte (↑),Intermedia,0.000000,Descartado (β = 0),0.969050,0.022886,Moderada/Baja (VIP < 1.0)
8,Salud_Joyeria,Uso de Joyería,Hábitos de Salud y Fuentes de Agua,Binaria,16.500000,U = 16.5,0.367359,0.937444,Atenuación (↓),Intermedia,-0.000000,Descartado (β = 0),0.629406,-0.021672,Moderada/Baja (VIP < 1.0)
9,Salud_Cualquier_Agua_Riesgo,Fuente de Agua no Tratada / Pozo,Otros Factores,Binaria,17.000000,U = 17.0,0.396212,0.937444,Atenuación (↓),Intermedia,-0.000000,Descartado (β = 0),0.456291,-0.005284,Moderada/Baja (VIP < 1.0)


Visualizamos la tabla con calidad de publicación mediante su renderizado interactivo HTML:

In [9]:
# Renderizado interactivo con estilos de publicación de HeavyStats
report

Determinante,Estadístico Bivariado,p (Crudo),q (FDR),β LASSO,Efecto LASSO,VIP PLS-R,Coef. PLS,Prioridad
Pescados,ρ = +0.523,0.017935002837647776,0.717400113505911,0.05999951908577348,Aporte (↑),1.3132325318525953,0.06753843059370068,Alta
Vehículo,U = 54.0,0.04181179117682194,0.8362358235364388,0.035465604861110146,Aporte (↑),1.2898884367029664,0.07271868320854646,Alta
Cereales,ρ = -0.371,0.10714264781858897,0.9374439244683992,-0.036148591789963345,Atenuación (↓),1.1430558798355703,-0.06058605655710297,Intermedia
Hábito de Fumar,U = 28.0,0.17712237049065838,0.9374439244683992,-0.005242097398858414,Atenuación (↓),1.1243449985556957,-0.07795574958633956,Intermedia
Transporte Público,U = 69.0,0.06784695360432105,0.904626048057614,0.017040274943253685,Aporte (↑),1.1085000431865684,0.06002929991174705,Intermedia
Taller Mecánico,U = 52.5,0.20478025275809542,0.9374439244683992,0.0,Descartado (β = 0),1.0150108890237401,0.0029088125185549544,Intermedia
Cualquier Taller en Entorno,U = 52.5,0.20478025275809542,0.9374439244683992,0.0,Descartado (β = 0),1.0150108890237401,0.0029088125185549544,Intermedia
Edad,ρ = +0.311,0.1814624366566317,0.9374439244683992,0.0,Descartado (β = 0),0.9690501056704732,0.022885853209268763,Intermedia
Uso de Joyería,U = 16.5,0.3673589785674638,0.9374439244683992,-0.0,Descartado (β = 0),0.6294061480449535,-0.021671764458697847,Intermedia
Fuente de Agua no Tratada / Pozo,U = 17.0,0.3962118754359596,0.9374439244683992,-0.0,Descartado (β = 0),0.456291001459173,-0.005284415845613359,Intermedia


---
## 5. Exportación Automática de Resultados (LaTeX, HTML, Excel)

Generamos los entregables editoriales listos para incluir directamente en el manuscrito de la tesis en LaTeX y para consulta interactiva.

In [10]:
salidas_dir = Path(r"d:\vscode\results\HeavyStats\notebooks\multivariante\salidas")
salidas_dir.mkdir(parents=True, exist_ok=True)

# 1. Exportar tabla en código LaTeX con booktabs
tex_file = salidas_dir / "tabla_multivariante_hg.tex"
latex_code = pipeline.export_latex(
    str(tex_file),
    caption="Tamizaje bivariante no paramétrico y parámetros de los modelos multivariantes (LASSO y PLS-R) para la concentración basal de mercurio total en sangre (\\ce{Hg_T}).",
    label="tab:multivariante_mercurio"
)

# 2. Exportar reporte HTML autónomo
html_file = salidas_dir / "reporte_multivariante_hg.html"
pipeline.export_html(str(html_file))

# 3. Exportar libro de Excel con hojas separadas
excel_file = salidas_dir / "resultados_multivariante_hg.xlsx"
with pd.ExcelWriter(excel_file) as writer:
    pipeline.screening_df.to_excel(writer, sheet_name="Cribado_Bivariante", index=False)
    pipeline.consolidated_df.to_excel(writer, sheet_name="Modelos_Multivariantes", index=False)

print(f"Archivos exportados exitosamente en: {salidas_dir}")
print(f"  • LaTeX: {tex_file.name} ({tex_file.stat().st_size} bytes)")
print(f"  • HTML:  {html_file.name} ({html_file.stat().st_size} bytes)")
print(f"  • Excel: {excel_file.name} ({excel_file.stat().st_size} bytes)")

Archivos exportados exitosamente en: d:\vscode\results\HeavyStats\notebooks\multivariante\salidas
  • LaTeX: tabla_multivariante_hg.tex (2653 bytes)
  • HTML:  reporte_multivariante_hg.html (14089 bytes)
  • Excel: resultados_multivariante_hg.xlsx (12351 bytes)


A continuación se muestra el código LaTeX generado automáticamente, listo para copiar o compilar con `\input{}`:

In [11]:
print(latex_code)

\begin{table}[htbp]
    \centering
    \small
    \caption{Tamizaje bivariante no paramétrico y parámetros de los modelos multivariantes (LASSO y PLS-R) para la concentración basal de mercurio total en sangre (\ce{Hg_T}).}
    \label{tab:multivariante_mercurio}
    \begin{tabular}{lcccccc}
        \toprule
        & \multicolumn{2}{c}{\textbf{Cribado Bivariante}} & \multicolumn{2}{c}{\textbf{Regresión LASSO}} & \multicolumn{2}{c}{\textbf{Regresión PLS-R}} \\
        \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}
        \textbf{Determinante / Factor} & \textbf{Estadístico} & \textbf{Valor $p$} & $\boldsymbol{\beta}$ \textbf{Estandarizado} & \textbf{Efecto} & \textbf{VIP} & \textbf{Coef. PLS} \\
        \midrule
        Pescados                             & $\rho_s = +0{,}523$ & 0{,}018  & +0{,}060   & Aporte       & 1{,}31   & +0{,}068 \\
        Vehículo                             & $U = 54{,}0$   & 0{,}042  & +0{,}035   & Aporte       & 1{,}29   & +0{,}073 \\
        Cere